<a href="https://colab.research.google.com/github/vishal9198/genAi-Labs/blob/main/simpleRAg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install "opentelemetry-api==1.42.1" "opentelemetry-sdk==1.42.1"

In [2]:
!pip install -q transformers sentence-transformers chromadb langchain-text-splitters torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [3]:
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
doc = """
Machine Learning is a subset of Artificial Intelligence that allows systems to learn from data.
Supervised learning uses labeled datasets to train algorithms to classify data or predict outcomes.
Common supervised algorithms include Linear Regression, Support Vector Machines, and Random Forests.
Unsupervised learning analyzes and clusters unlabeled datasets without human intervention.
Common unsupervised techniques include K-Means Clustering and Principal Component Analysis (PCA).
Reinforcement learning trains an agent to make a sequence of decisions by rewarding desired behaviors.
"""

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
chunks = splitter.split_text(doc)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
client = chromadb.Client()
db = client.create_collection(name="ml_docs")

In [8]:
for i, chunk in enumerate(chunks):
    vec = embed_model.encode(chunk).tolist()
    db.add(ids=[str(i)], embeddings=[vec], documents=[chunk])

print(f"Indexing complete! Total chunks stored: {len(chunks)}")

Indexing complete! Total chunks stored: 6


In [9]:
q = "What is reinforcement learning?"

# Convert query to vector and find top-2 most relevant chunks
q_vec = embed_model.encode(q).tolist()
res = db.query(query_embeddings=[q_vec], n_results=2)

In [10]:
retrieved_chunks = res["documents"][0]
context = " ".join(retrieved_chunks)

print("\n--- Retrieved Context ---")
print(context)


--- Retrieved Context ---
Reinforcement learning trains an agent to make a sequence of decisions by rewarding desired behaviors. Machine Learning is a subset of Artificial Intelligence that allows systems to learn from data.


In [13]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# 1. Load model and tokenizer directly
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 2. Construct Prompt: Context + Question
prompt = f"Answer the question based only on the context provided.\n\nContext: {context}\n\nQuestion: {q}\n\nAnswer:"

# 3. Tokenize input prompt
inputs = tokenizer(prompt, return_tensors="pt")

# 4. Generate output tokens directly with the model
outputs = model.generate(**inputs, max_new_tokens=64)

# 5. Decode tokens into text
ans = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n--- Generated Answer ---")
print(ans)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



--- Generated Answer ---
trains an agent to make a sequence of decisions by rewarding desired behaviors
